# Author: Victor Oluwabiyi
## Title: Axis A: Text Representation
**Depends on:** `../Data/cleaned_full_dataset.csv` — produced by Talha Khan

## What this notebook does
This notebook takes the cleaned arXiv cs.LG abstracts and converts them into three different numerical representations for downstream analysis by Chenyu Yang (comparison methods) and Sahil Bhinde (PCA and evaluation).

## Representations covered
1. **TF-IDF** — sparse matrix based on term frequency, used as the baseline.
2. **Word2Vec** — dense embeddings trained on the corpus using gensim.
3. **Sentence-BERT** — dense embeddings from a pre-trained transformer model.

## Files produced
| File | Location | To be used by |
|------|----------|---------|
| `tfidf_matrix.npz` | `../Data/` | Chenyu Yang, Sahil Bhinde |
| `tfidf_vocab.pkl` | `../Data/` | Chenyu Yang |
| `word2vec_embeddings.npy` | `../Data/` | Chenyu Yang, Sahil Bhinde |
| `sbert_embeddings.npy` | `../Data/` | Chenyu Yang, Sahil Bhinde |

# Loading The Dataset
First of all, all the libraries needed to run the notebook are imported. The cleaned dataset containing only **cs.LG** (Machine Learning) data is loaded and inspected to make sure everything is in order

In [23]:
import pandas as pd
import numpy as np
import scipy.sparse
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim.models import Word2Vec
from gensim.utils import tokenize
from sentence_transformers import SentenceTransformer

In [24]:
import numpy as np

# Load the dataset from the Data folder
df = pd.read_csv("../Data/arxiv_csLG_filtered.csv")

# Add period column matching the notebook's expected period names
df["period"] = np.where(df["year"] < 2010, "early",
                np.where(df["year"] < 2015, "mid",
                np.where(df["year"] < 2018, "growth", "recent")))

# Save split datasets per period (so the next section can load them)
df[df["period"] == "early"].to_csv("../Data/period_early.csv", index=False)
df[df["period"] == "mid"].to_csv("../Data/period_mid.csv", index=False)
df[df["period"] == "growth"].to_csv("../Data/period_growth.csv", index=False)
df[df["period"] == "recent"].to_csv("../Data/period_recent.csv", index=False)

# Inspect
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(list(df.columns))
print(df["period"].value_counts())

# Store for later use
docs = df["abstract"]
periods = df["period"]
years = df["year"]
print(f"\ndocs: {len(docs)} abstracts loaded")
print(f"periods: {periods.nunique()} unique periods")
print(f"years: {years.min()} - {years.max()}")

Shape: 7015 rows × 6 columns
['id', 'title', 'abstract', 'categories', 'year', 'period']
period
recent    4653
growth    1488
mid        724
early      150
Name: count, dtype: int64

docs: 7015 abstracts loaded
periods: 4 unique periods
years: 1999 - 2021


# Hypotheses for each Text Representation Approach

As mentioned above, these approaches below will be used to explore the text representation axis for the chosen dataset:

- TF-IDF
- Word2Vec
- Sentence-BERT

It is expected that the result of each approach will match the following hypotheses:
1. **TF-IDF:** It is expected that TF-IDF will focus more on keywords that distinguish one abstract from the others. Placing less emphasis on common words across different documents. Therefore I expect TF-IDF to clearly show a shift in vocabulary between periods, with modern terms appearing predominantly in later periods.
2. **Word2Vec:** It is expected that, unlike TF-IDF, Word2Vec will capture semantic relationships between terms, grouping similar concepts(e.g. synonyms) together even when different words are used. However since it is trained only on the arXiv corpus it may struggle with very rare or domain-specific terms that do not come up frequently. Therefore I expect Word2Vec to show clusters of similar methods or concepts from different periods even with the change in vocabulary.
3. **Sentence-BERT:** Since it is pre-trained on large general text rather than the corpus, it is expected that Sentence-BERT will produce the most semantically rich representations of the three approaches. Even with this advantage, it may underperform with highly technical and niche arXiv terminology that do not frequently appear in the pre-training data. Therefore I expect Sentence-BERT to produce a more gradual and consistent shift when abstracts are plotted over time, since it captures the overall meaning of a sentence(or abstract) rather than being thrown off by individual technical terms changing between periods.

# Representation 1: TF-IDF Matrix
TF-IDF represent each abstract as a vector of word importance scores. A term scores highly if it appears frequently in one abstract but rarely across all abstracts. This makes it good at identifying distinctive vocabulary per period. To capture domain-specific phrases such as 'neural network' and 'deep learning' as single features, bigrams are included alongside unigrams using ngram_range=(1,2). This is particularly important for ML abstracts where meaning often depends on word pairs rather than individual terms.

In [25]:
# preview the first abstract to check the text looks right before vectorizing
print(docs[0])

# cleaning using token_pattern which restricts to words only (3+ letters) to filter out numbers and noise from the abstracts
vectorizer = TfidfVectorizer(stop_words='english', ngram_range = (1, 2), token_pattern=r'[a-zA-Z]{3,}') #Run without max_features first to see the full vocabulary
vectorizer.fit(docs)

  This paper uncovers and explores the close relationship between Monte Carlo
Optimization of a parametrized integral (MCO), Parametric machine-Learning
(PL), and `blackbox' or `oracle'-based optimization (BO). We make four
contributions. First, we prove that MCO is mathematically identical to a broad
class of PL problems. This identity potentially provides a new application
domain for all broadly applicable PL techniques: MCO. Second, we introduce
immediate sampling, a new version of the Probability Collectives (PC) algorithm
for blackbox optimization. Immediate sampling transforms the original BO
problem into an MCO problem. Accordingly, by combining these first two
contributions, we can apply all PL techniques to BO. In our third contribution
we validate this way of improving BO by demonstrating that cross-validation and
bagging improve immediate sampling. Finally, conventional MC and MCO procedures
ignore the relationship between the sample point locations and the associated
values

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.","'[a-zA-Z]{3,}'"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","

We check the vectorizer to view the vocabulary created from the input text

In [26]:
vectorizer.vocabulary_

{'paper': 269292,
 'uncovers': 398037,
 'explores': 140134,
 'close': 60029,
 'relationship': 318248,
 'monte': 243457,
 'carlo': 49428,
 'optimization': 263695,
 'parametrized': 271567,
 'integral': 193941,
 'mco': 227804,
 'parametric': 271435,
 'machine': 221361,
 'learning': 210294,
 'blackbox': 42758,
 'oracle': 265016,
 'based': 35849,
 'make': 222631,
 'contributions': 78670,
 'prove': 302737,
 'mathematically': 226260,
 'identical': 178420,
 'broad': 45597,
 'class': 56262,
 'problems': 294582,
 'identity': 179187,
 'potentially': 285178,
 'provides': 303901,
 'new': 251863,
 'application': 19869,
 'domain': 113342,
 'broadly': 45680,
 'applicable': 19743,
 'techniques': 380826,
 'second': 339382,
 'introduce': 196916,
 'immediate': 180713,
 'sampling': 334888,
 'version': 412083,
 'probability': 292945,
 'collectives': 62959,
 'algorithm': 12144,
 'transforms': 392728,
 'original': 265917,
 'problem': 293229,
 'accordingly': 1866,
 'combining': 64058,
 'apply': 21343,
 'contri

In [27]:
# check the first 10 feature names. Note: this shows terms alphabetically not by importance
print(vectorizer.get_feature_names_out()[:10]) 

['aaa' 'aaa gaussian' 'aaa terms' 'aaai' 'aaai hydra' 'aaai ijcai'
 'aaai kvetonytm' 'aac' 'aac methods' 'aac significant']


In [28]:
# Transform the abstracts to a sparse TF-IDF matrix
tfidf_matrix = vectorizer.transform(docs)
print(tfidf_matrix.shape)

(7015, 420444)


In [29]:
# Get feature names and their mean TF-IDF scores across all documents
feature_names = vectorizer.get_feature_names_out()
mean_scores = np.asarray(tfidf_matrix.mean(axis=0)).flatten()

# sort by score from highest to lowest and print top 10
top_indices = mean_scores.argsort()[::-1][:10]
print([feature_names[i] for i in top_indices])

['learning', 'data', 'model', 'algorithm', 'models', 'training', 'based', 'neural', 'method', 'methods']


In [30]:
# save the sparse TF-IDF matrix 
scipy.sparse.save_npz("../Data/tfidf_matrix.npz", tfidf_matrix)

# save the vocabulary to map column indices back to words
import pickle
with open("../Data/tfidf_vocab.pkl", "wb") as f:
    pickle.dump(vectorizer.get_feature_names_out().tolist(), f)

print("tfidf_matrix.npz saved")
print("tfidf_vocab.pkl saved")

tfidf_matrix.npz saved
tfidf_vocab.pkl saved


# TF-IDF Insights: Top Terms per Period
The loaded dataset contains papers from years 1999-2021. Abstracts are sorted by their year into different periods marking different stages of research within the Machine Learning field:
- early (1999-2009)
- mid (2010-2014)
- growth (2015-2017)
- recent (2018-2021)

A separate TF-IDF is fitted on each period rather than the full corpus, so the scores reflect what is distinctive within that period alone

**Aim:** Looking at the most distinctive words per period will show how ML vocabulary has evolved between periods. 

In [31]:
print(periods.unique())

#Load the dataset from the files generated after period splitting
early = pd.read_csv("../Data/period_early.csv")
mid = pd.read_csv("../Data/period_mid.csv")
growth = pd.read_csv("../Data/period_growth.csv")
recent = pd.read_csv("../Data/period_recent.csv")

print(early.head()) #Check if dataset loaded properly

early_abstracts = early["abstract"]
mid_abstracts = mid["abstract"]
growth_abstracts = growth["abstract"]
recent_abstracts = recent["abstract"]

period_names = ['early', 'mid', 'growth', 'recent']
period_docs = [early_abstracts, mid_abstracts, growth_abstracts, recent_abstracts]

for i in range(len(period_names)):
    # fit a new TF-IDF on just this period's abstracts
    period_vectorizer = TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),
        token_pattern=r'[a-zA-Z]{3,}'
    )
    period_matrix = period_vectorizer.fit_transform(period_docs[i])

    # get top 15 terms by mean TF-IDF score
    feature_names = period_vectorizer.get_feature_names_out()
    mean_scores = np.asarray(period_matrix.mean(axis=0)).flatten()
    top_indices = mean_scores.argsort()[::-1][:15]
    
    print(f"\n--- {period_names[i]} ---")
    print([feature_names[i] for i in top_indices])

<StringArray>
['early', 'mid', 'growth', 'recent']
Length: 4, dtype: str
          id                                              title  \
0  0704.1274   Parametric Learning and Monte Carlo Optimization   
1  0704.2668  Supervised Feature Selection via Dependence Es...   
2  0705.1585  HMM Speaker Identification Using Linear and No...   
3  0706.3679  Scale-sensitive Psi-dimensions: the Capacity M...   
4  0707.3390  Consistency of the group Lasso and multiple ke...   

                                            abstract categories  year period  
0    This paper uncovers and explores the close r...  ['cs.LG']  2007  early  
1    We introduce a framework for filtering featu...  ['cs.LG']  2007  early  
2    Speaker identification is a powerful, non-in...  ['cs.LG']  2007  early  
3    Bounds on the risk play a crucial role in st...  ['cs.LG']  2007  early  
4    We consider the least-square regression prob...  ['cs.LG']  2007  early  

--- early ---
['learning', 'algorithm', 'data', '

# Count Matrix (Word Frequency Matrix)
LDA requires raw word counts rather than TF-IDF weighted values. This is because LDA is a probabilistic model that learns how words are distributed across topics i.e. it needs to know how often each word actually appears rather than a weighted score. The same settings are used as the TF-IDF vectorizer to keep the vocabulary consistent.

In [32]:
# build count vectorizer with same settings as TF-IDF to keep vocabulary consistent
count_vectorizer = CountVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    token_pattern=r'[a-zA-Z]{3,}'
)

# fit and transform to get raw word count matrix
count_matrix = count_vectorizer.fit_transform(docs)
print(count_matrix.shape)

# save as .npz for LDA topic model
scipy.sparse.save_npz("../Data/count_matrix.npz", count_matrix)
print("count_matrix.npz saved")

(7015, 420444)
count_matrix.npz saved


In [33]:
# Tokenize each abstract into a list of lowercase words
tokenized_docs = [list(tokenize(doc, lower=True)) for doc in docs]

# Train Word2Vec on the corpus (vector_size=100 is a sensible default)
w2v_model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=10,
    seed=42
)

print(f"Vocabulary size: {len(w2v_model.wv)}")
print(f"Vector size: {w2v_model.wv.vector_size}")

# Build document vectors by averaging word vectors in each abstract
def doc_to_vector(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.wv.vector_size)
    return np.mean(vectors, axis=0)

word2vec_embeddings = np.array([doc_to_vector(tokens, w2v_model) for tokens in tokenized_docs])
print(f"Word2Vec embeddings shape: {word2vec_embeddings.shape}")

# Save for teammates
np.save("../Data/word2vec_embeddings.npy", word2vec_embeddings)
print("word2vec_embeddings.npy saved")

Vocabulary size: 8662
Vector size: 100
Word2Vec embeddings shape: (7015, 100)
word2vec_embeddings.npy saved


# Representation 3: Sentence-BERT

Sentence-BERT uses a pre-trained transformer to produce sentence-level embeddings. Unlike Word2Vec (trained only on this corpus), Sentence-BERT brings external knowledge from large general text but may underperform on highly technical arXiv-specific terminology. Model: `all-MiniLM-L6-v2`, producing 384-dimensional embeddings.

In [34]:
sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

sbert_embeddings = sbert_model.encode(
    docs.tolist(),
    show_progress_bar=True,
    batch_size=32
)

print(f"Sentence-BERT embeddings shape: {sbert_embeddings.shape}")

np.save("../Data/sbert_embeddings.npy", sbert_embeddings)
print("sbert_embeddings.npy saved")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\oz25100\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\oz25100\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/220 [00:00<?, ?it/s]

Sentence-BERT embeddings shape: (7015, 384)
sbert_embeddings.npy saved
